In [ ]:
import pandas as pd

base = "../../../dataset/"

def load(path):
    # dtype=str keeps IDs as text; keep_default_na=False keeps empty cells as "" instead of NaN
    return pd.read_csv(base + path, sep="\t", dtype=str, keep_default_na=False)

s1 = load("train/train_source1.tsv")
s2 = load("train/train_source2.tsv")
s3 = load("train/train_source3.tsv")
gt = load("train/train_ground_truth.tsv")
t1 = load("test/test_source1.tsv")
t2 = load("test/test_source2.tsv")
t3 = load("test/test_source3.tsv")
print("loaded")

In [ ]:
for name, df in [("train S1", s1), ("train S2", s2), ("train S3", s3),
                 ("test S1", t1), ("test S2", t2), ("test S3", t3)]:
    print(f"{name}: {len(df):,} rows | countries: {df['country'].value_counts().to_dict()}")
print("ground truth rows:", len(gt))

In [ ]:
gt["ids"] = gt["matched_entity_ids"].apply(lambda x: [i for i in x.split(",") if i])
gt["n"] = gt["ids"].apply(len)

print("Singleton fraction (your 'do-nothing' score floor):", round((gt["n"] == 0).mean(), 3))
print("\nHow many matches per S1 entity:")
print(gt["n"].value_counts().sort_index().head(15))

pairs = gt.explode("ids").dropna(subset=["ids"])
print("\nMatches from S2 vs S3:", pairs["ids"].str[:2].value_counts().to_dict())

# Does any S2/S3 record get matched to more than one S1? (should be ~0)
print("S2/S3 IDs matched to >1 S1 entity:", (pairs["ids"].value_counts() > 1).sum())

# How many S2/S3 records match nothing at all?
matched = set(pairs["ids"])
print("Unmatched S2 fraction:", round(1 - s2["entity_id"].isin(matched).mean(), 3))
print("Unmatched S3 fraction:", round(1 - s3["entity_id"].isin(matched).mean(), 3))

In [ ]:
lookup = pd.concat([s1, s2, s3]).set_index("entity_id")
sample = pairs.sample(25, random_state=0)

for _, row in sample.iterrows():
    a, b = lookup.loc[row["source1_entity_id"]], lookup.loc[row["ids"]]
    print(f"[{a['country']}] {a['business_name']}  |  {a['business_address']}")
    print(f"   -> {b['business_name']}  |  {b['business_address']}\n")

## v3: full train (current model)

In [ ]:
import numpy as np, pandas as pd, gc
WORK, DATA = "../work", "../../../dataset"   # adjust if your paths differ
s1_tr  = pd.read_pickle(f"{WORK}/train_s1.pkl")
idx_tr = pd.read_pickle(f"{WORK}/train_idx.pkl")
cand   = pd.read_pickle(f"{WORK}/train_cand.pkl")

gt = pd.read_csv(f"{DATA}/train/train_ground_truth.tsv", sep="\t", dtype=str, keep_default_na=False)
true_tr = (gt.assign(ids=gt["matched_entity_ids"].str.split(","))
             .explode("ids").query("ids != ''")[["source1_entity_id", "ids"]])
s1p = pd.Series(np.arange(len(s1_tr)), index=s1_tr["entity_id"])
ixp = pd.Series(np.arange(len(idx_tr)), index=idx_tr["entity_id"])
truth = pd.DataFrame({"s1_pos": s1p[true_tr["source1_entity_id"]].to_numpy(),
                      "cand_pos": ixp[true_tr["ids"]].to_numpy()})
N = len(idx_tr)
tkey = truth["s1_pos"].to_numpy(np.int64) * N + truth["cand_pos"].to_numpy(np.int64)
print(f"{len(s1_tr):,} S1 | {N:,} S2/S3 | {len(cand):,} pairs | {len(truth):,} true pairs")

In [ ]:
# Skip: results known (recall 0.958). Needs `cand`, which the features cell deletes.
ckey = cand["s1_pos"].to_numpy(np.int64) * N + cand["cand_pos"].to_numpy(np.int64)
found = np.isin(tkey, ckey); del ckey
print(f"blocking recall: {found.mean():.4f}")
print(pd.Series(found).groupby(s1_tr["country"].to_numpy()[truth["s1_pos"]]).mean().round(4))
print("avg S1 competitors per S2/S3 (train):", round(cand.groupby("cand_pos").size().mean(), 2))

In [ ]:
from features import context_features, string_features, FEATURE_COLS, assign, f05_macro
ctx = context_features(cand, s1_tr, idx_tr)
del cand; gc.collect()
samp = ctx[(ctx["s1_pos"] % 20) < 3]
del ctx; gc.collect()
feat = string_features(samp, s1_tr, idx_tr, n_jobs=4)
del samp; gc.collect()
fkey = feat["s1_pos"].to_numpy(np.int64) * N + feat["cand_pos"].to_numpy(np.int64)
feat["label"] = np.isin(fkey, tkey)
is_val = (feat["s1_pos"] % 20) == 0
print(f"{len(feat):,} pairs | {feat['label'].mean():.3f} true | {is_val.mean():.2f} in val")

In [ ]:
print(len(FEATURE_COLS))

In [ ]:
# v4 training: run once, saves lgbm_v4.txt (lgbm_v3.txt is kept). Re-running overwrites lgbm_v4.txt.
import lightgbm as lgb
params = dict(objective="binary", learning_rate=0.1, num_leaves=63,
              min_child_samples=50, feature_fraction=0.8, verbose=-1)
dtr = lgb.Dataset(feat.loc[~is_val, FEATURE_COLS], feat.loc[~is_val, "label"])
dva = lgb.Dataset(feat.loc[is_val, FEATURE_COLS], feat.loc[is_val, "label"])
model = lgb.train(params, dtr, 1000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
model.save_model("lgbm_v4.txt")

In [ ]:
# Only needed in a fresh kernel instead of re-training (needs the v4 features cell above).
import lightgbm as lgb
model = lgb.Booster(model_file="lgbm_v4.txt")

In [ ]:
val = feat[is_val].copy()
val["prob"] = model.predict(val[FEATURE_COLS])
val_s1 = np.flatnonzero(np.arange(len(s1_tr)) % 20 == 0)
for th in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9]:
    m = assign(val, th)
    print(th, round(f05_macro(m, truth, val_s1), 4),
          f"| empty {1 - m['s1_pos'].nunique() / len(val_s1):.1%}")

## Error analysis (Sept 27)

In [ ]:
# Where do we lose F0.5 points on validation? Uses val, truth, val_s1, s1_tr, idx_tr, N, assign.
THR = 0.7
m = assign(val, THR)                                        # same decoding as the submission
P = m.groupby("s1_pos")["cand_pos"].agg(set).to_dict()      # predicted matches
C = val.groupby("s1_pos")["cand_pos"].agg(set).to_dict()    # blocking candidates
T = truth[truth["s1_pos"] % 20 == 0].groupby("s1_pos")["cand_pos"].agg(set).to_dict()  # true matches
owner = np.full(N, -1, np.int64)                            # true S1 of every S2/S3 record (-1 = none)
owner[truth["cand_pos"].to_numpy()] = truth["s1_pos"].to_numpy()
pkey = pd.Series(val["prob"].to_numpy(),
                 index=val["s1_pos"].to_numpy(np.int64) * N + val["cand_pos"].to_numpy(np.int64))
taken = set(m["cand_pos"])                                  # candidates assigned to some S1
country = s1_tr["country"].to_numpy()

def f05(p, t):
    if not t and not p: return 1.0
    tp = len(p & t)
    if tp == 0: return 0.0
    pr, rc = tp / len(p), tp / len(t)
    return 1.25 * pr * rc / (0.25 * pr + rc)

E, rows, fp_l, fn_l, bm_l = set(), [], [], [], []
for s in val_s1:
    t, p, c = T.get(s, E), P.get(s, E), C.get(s, E)
    fps, fns, bms = p - t, (t & c) - p, t - c
    fp_l += [(s, x) for x in fps]; fn_l += [(s, x) for x in fns]; bm_l += [(s, x) for x in bms]
    rows.append((s, country[s], len(t), f05(p, t), f05(t & c, t), f05(p & t, t),
                 f05(p | (t & c), t), len(fps), sum(owner[x] >= 0 for x in fps), len(fns), len(bms)))
ea = pd.DataFrame(rows, columns=["s1", "country", "n_true", "score", "ceiling", "no_fp",
                                 "no_fn", "fp", "fp_stolen", "fn", "blk_miss"])

mm = ea[["score", "ceiling", "no_fp", "no_fn"]].mean()
print(f"val F0.5 {mm.score:.4f} (should equal the sweep at 0.7) | ceiling given blocking {mm.ceiling:.4f}")
print(f"points lost -> FPs {mm.no_fp - mm.score:.4f} | FNs {mm.no_fn - mm.score:.4f} | blocking {1 - mm.ceiling:.4f}")
print(ea.groupby("country")[["score", "ceiling", "no_fp", "no_fn"]].mean().round(4))
sg = ea["n_true"] == 0
print(f"singletons {sg.mean():.1%} of val, score {ea.loc[sg, 'score'].mean():.3f} | others {ea.loc[~sg, 'score'].mean():.3f}")
print(f"FP {ea.fp.sum():,} (truly belong to another S1: {ea.fp_stolen.sum():,}) | "
      f"FN {len(fn_l):,} (lost to another S1 in assignment: {sum(x in taken for _, x in fn_l):,}) | "
      f"blocking misses {len(bm_l):,}")
fnp = pkey.reindex([s * N + x for s, x in fn_l]).to_numpy()
fpp = pkey.reindex([s * N + x for s, x in fp_l]).to_numpy()
print("FN prob quantiles 25/50/75/90%:", np.round(np.nanquantile(fnp, [.25, .5, .75, .9]), 3))
print("FP prob quantiles 10/25/50/75%:", np.round(np.nanquantile(fpp, [.1, .25, .5, .75]), 3))

In [ ]:
# Random examples of each error type (change the seed to see others)
# The full-train pickles keep only normalized text, so we show block_text (name + address).
print("S1 columns:", list(s1_tr.columns))
ta, ia = s1_tr["block_text"].to_numpy(), s1_tr["entity_id"].to_numpy()
tb, ib = idx_tr["block_text"].to_numpy(), idx_tr["entity_id"].to_numpy()
def show(s, c):
    print(f"[{country[s]}] {ia[s]}  {ta[s][:120]}")
    print(f"      {ib[c]}  {tb[c][:120]}")
    print(f"      prob={pkey.get(s * N + c, np.nan):.3f}   true S1: {ia[owner[c]] if owner[c] >= 0 else 'none'}\n")
rng = np.random.default_rng(0)
for title, lst in [("FALSE POSITIVES", fp_l), ("FALSE NEGATIVES", fn_l), ("BLOCKING MISSES", bm_l)]:
    print(f"========== {title} ({len(lst):,}) ==========")
    for k in rng.choice(len(lst), min(10, len(lst)), replace=False):
        show(*lst[k])

In [ ]:
# Do entity ID numbers carry matching information? (training files only, no external data)
num = lambda s: s.str.replace(r"\D", "", regex=True).astype(np.int64)
pp = true_tr.sample(500_000, random_state=0)
for src in ["S2", "S3"]:
    q = pp[pp["ids"].str.startswith(src)]
    print(src, "rank correlation S1 id vs matched id:",
          round(num(q["source1_entity_id"]).corr(num(q["ids"]), method="spearman"), 3))

In [ ]:
# How much of the loss involves S2/S3 records with an empty address?
emp = (idx_tr["addr_n"].str.strip() == "").to_numpy()
col = lambda lst: np.array([c for _, c in lst], dtype=np.int64)
print(f"empty address -> all S2/S3 {emp.mean():.1%} | predicted {emp[m['cand_pos'].to_numpy()].mean():.1%} | "
      f"FN {emp[col(fn_l)].mean():.1%} | FP {emp[col(fp_l)].mean():.1%} | blocking misses {emp[col(bm_l)].mean():.1%}")
nm = s1_tr["name_n"].to_numpy()
st = [(s, c) for s, c in fp_l if owner[c] >= 0]
print(f"stolen FPs where wrong S1 has the exact same name as the true S1: {np.mean([nm[s] == nm[owner[c]] for s, c in st]):.1%}")
fn_emp = [(s, c) for s, c in fn_l if emp[c]]
print(f"empty-address FNs: {len(fn_emp):,} of {len(fn_l):,} | median prob "
      f"{np.nanmedian(pkey.reindex([s * N + c for s, c in fn_emp]).to_numpy()):.3f}")